# CheXpert Reproduction + Robustness Extension (Kaggle GPU version)

This notebook reproduces the core methodology of **CheXpert: A Large Chest Radiograph Dataset with Uncertainty Labels and Expert Comparison** (Irvin et al., AAAI 2019), then extends it with a robustness-to-image-degradation experiment.

**Structure:**
- **Part 0 (Setup)** — Kaggle-specific: add the dataset via the UI, then locate files (no download/unzip needed, Kaggle mounts datasets directly)
- **Part 1 (Sections 1–7)** — reproduction: dataset pipeline, model, training, evaluation
- **Part 2 (Sections 8–15)** — extension: robustness to real-world image degradation


## Part 0: Environment Setup (Kaggle)

### Before running any code — in the Kaggle UI:

1. **Enable GPU**: on the right sidebar, go to **Settings → Accelerator → GPU T4 x2** (or P100 if offered). Kaggle provides ~30 free GPU-hours/week.
2. **Add the CheXpert dataset**: click **+ Add Input** (top right or right sidebar) → search **"chexpert"** → add the dataset (e.g. `ashery/chexpert`). It mounts read-only under `/kaggle/input/...` — no download or unzip step needed.
3. **To reuse a previously-trained checkpoint** instead of retraining from scratch:
   - Upload the `.pth` checkpoint file(s) to Kaggle via **kaggle.com/datasets → New Dataset**, giving it a name (e.g. `chexpert-checkpoints`)
   - Back in this notebook, click **+ Add Input** again → search for the checkpoint dataset → add it. It will appear under `/kaggle/input/chexpert-checkpoints/`
   - Otherwise, run Part 1 (training) directly on Kaggle's GPU — a fresh 50–100k run is typically fast enough on Kaggle's free tier.


### Locate the dataset files

Kaggle mounts datasets under `/kaggle/input/<dataset-name>/...`, but the exact folder name can vary. This searches for `train.csv` automatically and sets `DATA_ROOT` to wherever it's actually found, avoiding the need to guess or hardcode a path that might not match.

In [ ]:
import os

DATA_ROOT = None
for root, dirs, filenames in os.walk('/kaggle/input'):
    if 'train.csv' in filenames:
        DATA_ROOT = root
        break

if DATA_ROOT is None:
    raise FileNotFoundError(
        "Could not find train.csv under /kaggle/input. "
        "Make sure you've added the CheXpert dataset via '+ Add Input' in the sidebar."
    )

print(f"Found dataset at: {DATA_ROOT}")
print("Contents:", os.listdir(DATA_ROOT))


### Inspect the label structure

CheXpert's labels use: `1.0` = positive, `0.0` = negative, `-1.0` = uncertain, `NaN` = pathology not mentioned in the radiology report at all. This is the automatically-generated output of the paper's rule-based labeler (see the paper's Section 3) — we use these labels as-is rather than rebuilding the labeler ourselves.

In [ ]:
import pandas as pd

df = pd.read_csv(f'{DATA_ROOT}/train.csv')
print(df.shape)
print(df.columns.tolist())
df.head()


### Locate an existing checkpoint dataset (only needed when reusing a previously-trained model)

This cell can be skipped when training fresh on Kaggle — new checkpoints are generated in Section 7 instead.

In [ ]:
import glob

# Auto-detect any .pth files under /kaggle/input (your uploaded checkpoint dataset)
CHECKPOINT_SEARCH_DIR = '/kaggle/input'
found_checkpoints = sorted(glob.glob(f'{CHECKPOINT_SEARCH_DIR}/**/*.pth', recursive=True))
print("Found checkpoints:", found_checkpoints)


### Output directory

Kaggle notebooks write to `/kaggle/working/`, which persists for the current session and is saved automatically on "Save Version." No separate Drive mount is needed, but `/kaggle/working` has a size limit (usually a few GB), so large intermediate files should be avoided there.

In [ ]:
import os
os.makedirs('/kaggle/working/checkpoints', exist_ok=True)
os.makedirs('/kaggle/working/results', exist_ok=True)
print("Output directories ready.")


## Part 1: Reproduction — All 5 Uncertainty-Handling Strategies

The paper's actual core methodology isn't just "train one DenseNet121" — it's a **comparison of 5 different ways of handling the `-1` (uncertain) label**: U-Ignore, U-Zeros, U-Ones, U-SelfTrained, and U-MultiClass. They train all 5, evaluate each per pathology, then build their **final model as a per-pathology composite** — using whichever strategy scored best for each individual pathology, rather than one single "winning" strategy overall.

This section reproduces that full process: all 5 strategies, trained on the **same fixed subsample of images** (so the comparison isolates the effect of label-handling, not random sampling differences), evaluated per pathology, and combined into a final composite model — matching the paper's actual final-model construction.


### 1. Shared dataset class + a fixed training subsample

`ImageLabelDataset` is a generic loader: given a dataframe with labels already prepared (however a given strategy needs them — binary 0/1, soft probabilities, or 3-way class indices), it loads the image and returns it with the corresponding label tensor. This one class serves every strategy; only the *label values* going into `df` differ between them.

We fix **one subsample of training images** (`raw_train_df`, sampled once with a fixed seed) and derive all 5 strategies' labels from this same set of images — this way, any AUC differences between strategies reflect the label-handling method itself, not which images happened to get sampled.

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision.models as models
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler
from sklearn.metrics import roc_auc_score
from PIL import Image
import os

PATHOLOGIES = ['Atelectasis', 'Cardiomegaly', 'Consolidation', 'Edema', 'Pleural Effusion']
BATCH_SIZE = 16
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

train_transform = transforms.Compose([
    transforms.Resize((320, 320)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])
val_transform = transforms.Compose([
    transforms.Resize((320, 320)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

class ImageLabelDataset(Dataset):
    """Generic loader: labels are already prepared in `df` — this class just loads images and returns them alongside whatever label values/dtype the strategy needs."""
    def __init__(self, df, data_root, pathologies, transform, label_dtype=torch.float32):
        self.df = df.reset_index(drop=True)
        self.data_root = data_root
        self.pathologies = pathologies
        self.transform = transform
        self.label_dtype = label_dtype

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        relative_path = row['Path'].replace('CheXpert-v1.0-small/', '')
        img_path = f"{self.data_root}/{relative_path}"
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        np_dtype = np.int64 if self.label_dtype == torch.long else np.float32
        labels = torch.tensor(row[self.pathologies].values.astype(np_dtype), dtype=self.label_dtype)
        return image, labels

# ---- Fixed training subsample (shared across all 5 strategies) ----
N_TRAIN_STRATEGY = 40000  # adjust based on your time budget; 5 strategies will each train on this many images
N_VAL = 200  # official valid.csv, ~234 images total, kept as-is

full_train_df = pd.read_csv(f'{DATA_ROOT}/train.csv')
full_train_df = full_train_df[full_train_df['Frontal/Lateral'] == 'Frontal'].reset_index(drop=True)
for p in PATHOLOGIES:
    full_train_df[p] = full_train_df[p].fillna(0)  # not-mentioned -> negative, consistent across all 5 strategies

raw_train_df = full_train_df.sample(n=min(N_TRAIN_STRATEGY, len(full_train_df)), random_state=42).reset_index(drop=True)
print(f"Fixed training subsample: {len(raw_train_df)} images (shared across all 5 strategies)")
print("Uncertain (-1) label rate per pathology in this subsample:")
print((raw_train_df[PATHOLOGIES] == -1).mean())

# ---- Validation set: always the paper's real official validation labels (no uncertainty there) ----
val_df = pd.read_csv(f'{DATA_ROOT}/valid.csv')
val_df = val_df[val_df['Frontal/Lateral'] == 'Frontal'].reset_index(drop=True)
for p in PATHOLOGIES:
    val_df[p] = val_df[p].fillna(0)

val_dataset = ImageLabelDataset(val_df, DATA_ROOT, PATHOLOGIES, val_transform, label_dtype=torch.float32)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
print(f"Validation size: {len(val_dataset)}")


### 2. Model builders, loss functions, and training/evaluation helpers

Four of the five strategies (U-Zeros, U-Ones, U-Ignore, U-SelfTrained) are standard binary multi-label classifiers — same DenseNet121 architecture, just different label values or loss masking:
- **U-Zeros / U-Ones**: uncertain (-1) labels remapped to 0 or 1 before training — standard `BCEWithLogitsLoss`
- **U-Ignore**: uncertain labels become `NaN`; a **masked loss** zeroes out their contribution entirely, so the model only learns from confidently-labeled examples
- **U-SelfTrained**: trained in two stages — first a U-Ignore model, then its predicted probabilities replace the uncertain labels as soft targets for a second training pass (built in Section 5 below)

**U-MultiClass** is architecturally different: instead of predicting one probability per pathology, it predicts a 3-way distribution (negative / positive / uncertain) per pathology, trained with cross-entropy. At evaluation time, we follow the paper's approach — restrict the softmax to just the negative/positive classes to get a comparable positive-probability for AUC scoring.

In [ ]:
def build_model(num_classes=len(PATHOLOGIES)):
    """Standard binary multi-label DenseNet121 — used for U-Zeros, U-Ones, U-Ignore, U-SelfTrained."""
    model = models.densenet121(weights='DEFAULT')
    num_features = model.classifier.in_features
    model.classifier = nn.Linear(num_features, num_classes)
    return model.to(device)

def build_model_multiclass(num_classes=len(PATHOLOGIES)):
    """DenseNet121 with a 3-way output per pathology (negative / positive / uncertain) for U-MultiClass."""
    model = models.densenet121(weights='DEFAULT')
    num_features = model.classifier.in_features
    model.classifier = nn.Linear(num_features, num_classes * 3)
    return model.to(device)

scaler = GradScaler('cuda')
bce_criterion = nn.BCEWithLogitsLoss()

def masked_bce_loss(outputs, labels):
    """BCE loss that ignores NaN-labeled (uncertain) entries entirely — used for U-Ignore."""
    mask = ~torch.isnan(labels)
    labels_filled = torch.where(mask, labels, torch.zeros_like(labels))
    loss_elem = F.binary_cross_entropy_with_logits(outputs, labels_filled, reduction='none')
    return (loss_elem * mask.float()).sum() / mask.float().sum().clamp(min=1.0)

def train_one_epoch_generic(model, loader, optimizer, loss_fn, device):
    model.train()
    total_loss = 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        with autocast('cuda'):
            outputs = model(images)
            loss = loss_fn(outputs, labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item()
    return total_loss / len(loader)

def train_one_epoch_multiclass(model, loader, optimizer, device, num_pathologies):
    model.train()
    total_loss = 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)  # labels: (batch, num_path) long, values in {0,1,2}
        optimizer.zero_grad()
        with autocast('cuda'):
            outputs = model(images).view(-1, num_pathologies, 3).permute(0, 2, 1)  # (batch, 3, num_path)
            loss = F.cross_entropy(outputs, labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item()
    return total_loss / len(loader)

def evaluate(model, loader, device, pathologies):
    """Standard AUC evaluation for binary multi-label models."""
    model.eval()
    all_labels, all_preds = [], []
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            outputs = torch.sigmoid(model(images))
            all_preds.append(outputs.cpu().numpy())
            all_labels.append(labels.numpy())
    all_preds = np.concatenate(all_preds)
    all_labels = np.concatenate(all_labels)
    aucs = {}
    for i, p in enumerate(pathologies):
        try:
            aucs[p] = roc_auc_score(all_labels[:, i], all_preds[:, i])
        except ValueError:
            aucs[p] = float('nan')
    return aucs

def evaluate_multiclass(model, loader, device, pathologies):
    """AUC evaluation for U-MultiClass: derive positive-probability via softmax restricted to {negative, positive}."""
    model.eval()
    all_labels, all_probs = [], []
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            outputs = model(images).view(-1, len(pathologies), 3)
            logit_neg, logit_pos = outputs[:, :, 0], outputs[:, :, 1]
            p_pos = torch.softmax(torch.stack([logit_neg, logit_pos], dim=-1), dim=-1)[..., 1]
            all_probs.append(p_pos.cpu().numpy())
            all_labels.append(labels.numpy())
    all_probs = np.concatenate(all_probs)
    all_labels = np.concatenate(all_labels)
    aucs = {}
    for i, p in enumerate(pathologies):
        try:
            aucs[p] = roc_auc_score(all_labels[:, i], all_probs[:, i])
        except ValueError:
            aucs[p] = float('nan')
    return aucs

os.makedirs('/kaggle/working/checkpoints', exist_ok=True)

def run_training(strategy_name, dataset, num_epochs, loss_fn, save_prefix):
    model = build_model()
    optimizer = optim.Adam(model.parameters(), lr=1e-4)
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
    history = []
    for epoch in range(num_epochs):
        train_loss = train_one_epoch_generic(model, loader, optimizer, loss_fn, device)
        aucs = evaluate(model, val_loader, device, PATHOLOGIES)
        print(f"[{strategy_name}] Epoch {epoch+1}/{num_epochs} — Loss: {train_loss:.4f}")
        for p, a in aucs.items():
            print(f"    {p}: {a:.4f}")
        history.append({'epoch': epoch + 1, 'loss': train_loss, **aucs})
        torch.save(model.state_dict(), f'/kaggle/working/checkpoints/{save_prefix}_e{epoch+1}.pth')
    final_aucs = evaluate(model, val_loader, device, PATHOLOGIES)
    return model, final_aucs, pd.DataFrame(history)

NUM_EPOCHS = 3  # matches the paper's exact epoch count


### 3. Train U-Zeros and U-Ones

The two simplest strategies — uncertain labels remapped to a fixed 0 or 1 before training.

In [ ]:
# --- U-Zeros ---
df_zeros = raw_train_df.copy()
for p in PATHOLOGIES:
    df_zeros[p] = df_zeros[p].replace(-1, 0)
dataset_zeros = ImageLabelDataset(df_zeros, DATA_ROOT, PATHOLOGIES, train_transform)

model_zeros, aucs_zeros, history_zeros = run_training('U-Zeros', dataset_zeros, NUM_EPOCHS, bce_criterion, 'zeros')

print("\nFinal U-Zeros AUCs:", aucs_zeros)


In [ ]:
# --- U-Ones ---
df_ones = raw_train_df.copy()
for p in PATHOLOGIES:
    df_ones[p] = df_ones[p].replace(-1, 1)
dataset_ones = ImageLabelDataset(df_ones, DATA_ROOT, PATHOLOGIES, train_transform)

model_ones, aucs_ones, history_ones = run_training('U-Ones', dataset_ones, NUM_EPOCHS, bce_criterion, 'ones')

print("\nFinal U-Ones AUCs:", aucs_ones)


### 4. Train U-Ignore

Uncertain labels become `NaN`, and the masked loss function excludes them from training entirely — the model only ever learns from confidently positive/negative examples.

In [ ]:
# --- U-Ignore ---
df_ignore = raw_train_df.copy()
for p in PATHOLOGIES:
    df_ignore[p] = df_ignore[p].replace(-1, np.nan)
dataset_ignore = ImageLabelDataset(df_ignore, DATA_ROOT, PATHOLOGIES, train_transform)

model_ignore, aucs_ignore, history_ignore = run_training('U-Ignore', dataset_ignore, NUM_EPOCHS, masked_bce_loss, 'ignore')

print("\nFinal U-Ignore AUCs:", aucs_ignore)


### 5. Train U-SelfTrained (bootstrapped from the U-Ignore model)

This is a two-stage strategy: we already have a trained U-Ignore model from Section 4. We now use it to *predict* probabilities for every originally-uncertain (-1) training example, and use those predictions as soft training targets (a continuous value between 0 and 1) instead of a hard label — while true 0/1 labels stay untouched everywhere else. A fresh model is then trained on this relabeled dataset with standard BCE loss (which handles soft float targets fine).

In [ ]:
# Step 1: use the trained U-Ignore model to predict soft labels for originally-uncertain entries
uncertain_mask_df = (raw_train_df[PATHOLOGIES] == -1)

inference_df = raw_train_df.copy()
for p in PATHOLOGIES:
    inference_df[p] = inference_df[p].replace(-1, 0)  # placeholder, not used — we only need images here
inference_dataset = ImageLabelDataset(inference_df, DATA_ROOT, PATHOLOGIES, val_transform)  # no augmentation for inference
inference_loader = DataLoader(inference_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

model_ignore.eval()
all_soft_preds = []
with torch.no_grad():
    for images, _ in inference_loader:
        images = images.to(device)
        preds = torch.sigmoid(model_ignore(images)).cpu().numpy()
        all_soft_preds.append(preds)
all_soft_preds = np.concatenate(all_soft_preds)

df_selftrained = raw_train_df.copy()
for i, p in enumerate(PATHOLOGIES):
    col_mask = uncertain_mask_df[p].values
    df_selftrained.loc[col_mask, p] = all_soft_preds[col_mask, i]

print(f"Replaced {uncertain_mask_df.values.sum()} originally-uncertain label entries with soft predictions.")

# Step 2: train a fresh model on this relabeled data
dataset_selftrained = ImageLabelDataset(df_selftrained, DATA_ROOT, PATHOLOGIES, train_transform)
model_selftrained, aucs_selftrained, history_selftrained = run_training(
    'U-SelfTrained', dataset_selftrained, NUM_EPOCHS, bce_criterion, 'selftrained'
)

print("\nFinal U-SelfTrained AUCs:", aucs_selftrained)


### 6. Train U-MultiClass

Architecturally different from the other four: instead of one probability per pathology, the model outputs a 3-way distribution (negative / positive / uncertain) per pathology, trained with cross-entropy. This lets the model learn its own internal representation of genuine ambiguity rather than being forced into a binary decision on uncertain cases.

In [ ]:
df_multiclass = raw_train_df.copy()
for p in PATHOLOGIES:
    # 0 = negative, 1 = positive, 2 = uncertain
    df_multiclass[p] = df_multiclass[p].replace(-1, 2).astype(int)

dataset_multiclass = ImageLabelDataset(df_multiclass, DATA_ROOT, PATHOLOGIES, train_transform, label_dtype=torch.long)
loader_multiclass = DataLoader(dataset_multiclass, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)

model_multiclass = build_model_multiclass()
optimizer_mc = optim.Adam(model_multiclass.parameters(), lr=1e-4)

history_multiclass = []
for epoch in range(NUM_EPOCHS):
    train_loss = train_one_epoch_multiclass(model_multiclass, loader_multiclass, optimizer_mc, device, len(PATHOLOGIES))
    aucs = evaluate_multiclass(model_multiclass, val_loader, device, PATHOLOGIES)
    print(f"[U-MultiClass] Epoch {epoch+1}/{NUM_EPOCHS} — Loss: {train_loss:.4f}")
    for p, a in aucs.items():
        print(f"    {p}: {a:.4f}")
    history_multiclass.append({'epoch': epoch + 1, 'loss': train_loss, **aucs})
    torch.save(model_multiclass.state_dict(), f'/kaggle/working/checkpoints/multiclass_e{epoch+1}.pth')

aucs_multiclass = evaluate_multiclass(model_multiclass, val_loader, device, PATHOLOGIES)
history_multiclass = pd.DataFrame(history_multiclass)
print("\nFinal U-MultiClass AUCs:", aucs_multiclass)


### 7. Compare all 5 strategies — reproducing the paper's Table 3

This is the paper's actual core result: a per-pathology comparison across all 5 uncertainty-handling strategies. The best strategy varies by pathology in the paper (e.g. U-Ones wins on Atelectasis, U-MultiClass wins on Cardiomegaly) — we check whether our results show similar patterns.

In [ ]:
comparison_df = pd.DataFrame({
    'U-Zeros': aucs_zeros,
    'U-Ones': aucs_ones,
    'U-Ignore': aucs_ignore,
    'U-SelfTrained': aucs_selftrained,
    'U-MultiClass': aucs_multiclass,
}).T

comparison_df = comparison_df[PATHOLOGIES]
print("=== AUC by Strategy and Pathology ===\n")
print(comparison_df.round(4).to_string())

best_strategy_per_pathology = comparison_df.idxmax(axis=0)
print("\n=== Best Strategy per Pathology ===")
for p in PATHOLOGIES:
    print(f"  {p}: {best_strategy_per_pathology[p]} (AUC = {comparison_df.loc[best_strategy_per_pathology[p], p]:.4f})")

comparison_df.to_csv('/kaggle/working/results/strategy_comparison.csv')


### 8. Build the final composite model — best strategy per pathology

Matching the paper's actual final-model construction: rather than picking one overall "winning" strategy, we build a composite model that uses whichever strategy's model scored best **for each individual pathology**. `CompositeBestModel` wraps the underlying trained models and, on each forward pass, only runs the unique models actually needed (not redundantly), then assembles the final prediction per pathology from the right source model. It returns logits in the same shape/format as a normal single model, so it's a drop-in replacement anywhere downstream (evaluation, robustness testing) expects `model(images)`.

In [ ]:
strategy_models = {
    'U-Zeros': model_zeros,
    'U-Ones': model_ones,
    'U-Ignore': model_ignore,
    'U-SelfTrained': model_selftrained,
    'U-MultiClass': model_multiclass,
}

best_per_pathology = {p: (best_strategy_per_pathology[p], strategy_models[best_strategy_per_pathology[p]]) for p in PATHOLOGIES}

class CompositeBestModel(nn.Module):
    """Routes each pathology's prediction to whichever strategy's model scored best for it,
    matching the paper's own final-model construction. Returns logits shaped (batch, num_pathologies),
    so it's a drop-in replacement wherever a normal single model is expected."""
    def __init__(self, best_per_pathology, pathologies):
        super().__init__()
        self.pathologies = pathologies
        self.best_per_pathology = best_per_pathology
        self.unique_models = nn.ModuleDict()
        for p, (strat, m) in best_per_pathology.items():
            if strat not in self.unique_models:
                self.unique_models[strat] = m

    def forward(self, x):
        raw_outputs = {strat: m(x) for strat, m in self.unique_models.items()}
        batch = x.size(0)
        final_logits = torch.zeros(batch, len(self.pathologies), device=x.device)
        for i, p in enumerate(self.pathologies):
            strat, _ = self.best_per_pathology[p]
            out = raw_outputs[strat]
            if strat == 'U-MultiClass':
                out_reshaped = out.view(batch, len(self.pathologies), 3)
                logit_neg, logit_pos = out_reshaped[:, i, 0], out_reshaped[:, i, 1]
                p_pos = torch.softmax(torch.stack([logit_neg, logit_pos], dim=-1), dim=-1)[:, 1]
                p_pos = p_pos.clamp(1e-6, 1 - 1e-6)
                final_logits[:, i] = torch.log(p_pos / (1 - p_pos))
            else:
                final_logits[:, i] = out[:, i]
        return final_logits

composite_model = CompositeBestModel(best_per_pathology, PATHOLOGIES).to(device)
composite_model.eval()

composite_aucs = evaluate(composite_model, val_loader, device, PATHOLOGIES)
print("=== Final Composite Model AUCs (paper-style best-per-pathology) ===")
for p, a in composite_aucs.items():
    print(f"  {p}: {a:.4f}")

comparison_df.loc['Composite (final)'] = pd.Series(composite_aucs)
comparison_df.to_csv('/kaggle/working/results/strategy_comparison.csv')
print("\nUpdated comparison table saved.")


### Checkpoint persistence note

All 5 strategies' per-epoch checkpoints are saved to `/kaggle/working/checkpoints/`. This persists for the current session and is saved automatically on **Save Version**. The `composite_model` object itself (in memory) is what carries forward into the robustness extension below — no need to reload anything from disk.

In [ ]:
print(sorted(os.listdir('/kaggle/working/checkpoints')))


## ✅ Session 1 Complete

**Save progress before continuing: click "Save Version" (top right) → "Save & Run All (Commit)".**

This commits all 5 trained model checkpoints (`/kaggle/working/checkpoints/`) and the strategy comparison table (`/kaggle/working/results/strategy_comparison.csv`) as this notebook's permanent output.

### Next steps — second-seed training:
1. Once this version finishes committing, open **`LebNet_Session1b_SecondSeed.ipynb`**
2. In that notebook, click **+ Add Input** → search for this notebook by name (under "Your Work" / "Notebooks") → add it
3. Its checkpoints will be available at `/kaggle/input/<this-notebook-slug>/checkpoints/...`
4. Session 1b reloads these checkpoints and continues from here — no retraining needed

Training and robustness testing are split across separate notebooks because the full pipeline can exceed Kaggle's ~9-hour single-session limit.